# Dataset Preparation for Fine-Tuning

This notebook prepares training data for fine-tuning by enriching questions with context and generating high-quality answers.

## Pipeline Stages

**Stage 3: Context Pairing**  
For each question, we bundle the relevant golden context with carefully sampled distractor contexts to create training pairs.

**Stage 4: Answer Generation**  
Generate factual answers to each question using strictly the golden context, ensuring accuracy and grounding.

**Stage 5: Dataset Export**  
Join all components and export the complete dataset (question, answer, positive_ctx, negative_ctxs) to a JSONL file ready for model training.

---

**Inputs:** Delta tables containing `chunks` and `questions`  
**Outputs:** Delta tables for `gd_pairs` and `answers`, plus a JSONL dataset file

## Setup: Install Required Libraries

Installing the necessary Python packages for LLM interaction, data processing, and retry logic.

In [0]:
%pip install langchain openai tenacity pyspark
dbutils.library.restartPython()

## API Configuration

Configuring NVIDIA NIM API credentials for LLM access. In production environments, always use secret management instead of hardcoded keys.

In [0]:
# Replace with your own key from https://build.nvidia.com
api = "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"
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
NVIDIA_MODEL = "openai/gpt-oss-120b"

# Production note: Use dbutils.secrets.get() instead of hardcoded keys

## Import Libraries

Importing all necessary libraries for data processing, LLM API calls, and concurrent execution.

In [0]:
import os
import json
import uuid
import random
import logging
import re
import threading
import hashlib
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, TimestampType
import pyspark.sql.functions as F

import openai
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

spark = SparkSession.builder.getOrCreate()

## Configuration

Defining paths, parameters, and settings for the entire pipeline including:
* Unity Catalog locations for data storage
* Chunking and sampling parameters
* API rate limits and batch processing configuration

In [0]:
CATALOG = "workspace"
SCHEMA  = "default"
VOLUME  = "code_v1"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

PATHS = {
    "inbox_csv":      f"{BASE}/bronze/inbox/CSV",
    "inbox_pdf":      f"{BASE}/bronze/inbox/PDF",
    "archive_csv":    f"{BASE}/bronze/inbox/outbox/archive/CSV",
    "archive_pdf":    f"{BASE}/bronze/inbox/outbox/archive/PDF",
    "chunks":         f"{BASE}/bronze/chunks",
    "questions":      f"{BASE}/bronze/questions",
    "gd_pairs":       f"{BASE}/bronze/gd_pairs",
    "answers":        f"{BASE}/bronze/answers",
    "datasets":       f"{BASE}/silver/pub/datasets",
    "manifests":      f"{BASE}/bronze/_manifests",
    "logs":           f"{BASE}/bronze/_logs",
}

NUM_DISTRACTORS = 3
NAMESPACE = uuid.NAMESPACE_OID

# Chunking parameters
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
QUESTIONS_PER_CHUNK = 3

# API rate limiting and batching
API_RATE_LIMIT = 30          # calls per minute (max parallel calls per wave)
BATCH_SIZE = 20              # chunks per API call (10 chunks = 30 answers per call)
FALLBACK_BATCH_SIZE = 5      # fallback size if batches fail
MIN_WAVE_INTERVAL = 60.0     # seconds between waves (1 minute for rate limit)
MAX_MODEL_TOKENS = 90000     # Model's max context window

## Logging & Directory Setup

Setting up logging infrastructure and ensuring all required directories exist. Manifest files track processing progress to enable resumable execution.

In [0]:
def setup_logging():
    logger = logging.getLogger("dataset_prep")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        ch.setFormatter(formatter)
        logger.addHandler(ch)
    return logger

logger = setup_logging()

def ensure_dir(path: str):
    try:
        dbutils.fs.mkdirs(path)
    except Exception as e:
        logger.error(f"Failed to create directory {path}: {e}")
        raise

for key, path in PATHS.items():
    ensure_dir(path)

def load_manifest(name: str) -> dict:
    manifest_path = f"{PATHS['manifests']}/{name}.json"
    if os.path.exists(manifest_path):
        with open(manifest_path, "r") as f:
            return json.load(f)
    return {}

def save_manifest(name: str, entries: dict):
    manifest_path = f"{PATHS['manifests']}/{name}.json"
    with open(manifest_path, "w") as f:
        json.dump(entries, f, indent=2)

## Load Base Data

Loading chunks and questions from Delta tables into memory for fast lookup during processing.

In [0]:
try:
    df_chunks = spark.read.format("delta").load(PATHS["chunks"])
    df_questions = spark.read.format("delta").load(PATHS["questions"])
    
    chunks_local = df_chunks.select("chunk_id", "doc_id", "chunk_text").collect()
    questions_local = df_questions.select("question_id", "chunk_id", "doc_id", "question_text").collect()
    
    chunk_id_to_text = {row.chunk_id: row.chunk_text for row in chunks_local}
    doc_to_chunks = {}
    for row in chunks_local:
        doc_to_chunks.setdefault(row.doc_id, []).append(row.chunk_id)
        
    all_chunk_ids = list(chunk_id_to_text.keys())
    
    logger.info(f"Cached {len(chunk_id_to_text)} chunks and {len(questions_local)} questions.")
except Exception as e:
    logger.error(f"Required base tables are missing. Stop execution. Error: {e}")
    dbutils.notebook.exit("Missing base Delta tables")

## Distractor Sampling Strategy

Implementing deterministic sampling of negative contexts (distractors) for each question. We preferentially sample from different documents to ensure meaningful contrast.

In [0]:
def sample_distractors(golden_chunk_id: str, golden_doc_id: str, k: int) -> list:
    # Use deterministic seeding for reproducible results
    seed = int(hashlib.md5((golden_chunk_id + str(k)).encode('utf-8')).hexdigest(), 16) % (2**32)
    rng = random.Random(seed)
    
    # Prefer chunks from different documents for better contrast
    pool = [cid for cid in all_chunk_ids if cid not in doc_to_chunks.get(golden_doc_id, [])]
    
    if len(pool) < k:
        # Fall back to same-document chunks if needed
        pool = [cid for cid in all_chunk_ids if cid != golden_chunk_id]
        
    if len(pool) < k:
        return pool
        
    return rng.sample(pool, k)

## Build Question-Context Pairs

Creating training pairs by matching each question with its golden context and sampled distractor contexts. Progress is tracked via manifest to support resumable execution.

In [0]:
gd_pairs_schema = StructType([
    StructField("pair_id", StringType(), False),
    StructField("question_id", StringType(), False),
    StructField("chunk_id", StringType(), False),
    StructField("golden_text", StringType(), False),
    StructField("distractor_chunk_ids", ArrayType(StringType()), False),
    StructField("distractor_texts", ArrayType(StringType()), False),
    StructField("created_at_utc", TimestampType(), False)
])

gd_manifest = load_manifest("processed_questions_gd")
pending_questions = [q for q in questions_local if q.question_id not in gd_manifest]

logger.info(f"Found {len(pending_questions)} questions needing distractor pairing.")

new_pairs = []
now_utc = datetime.now(timezone.utc)

for q in pending_questions:
    pair_id = str(uuid.uuid5(NAMESPACE, f"pair:{q.question_id}"))
    golden_text = chunk_id_to_text.get(q.chunk_id, "")
    
    distractor_ids = sample_distractors(q.chunk_id, q.doc_id, NUM_DISTRACTORS)
    distractor_texts = [chunk_id_to_text[did] for did in distractor_ids]
    
    new_pairs.append({
        "pair_id": pair_id,
        "question_id": q.question_id,
        "chunk_id": q.chunk_id,
        "golden_text": golden_text,
        "distractor_chunk_ids": distractor_ids,
        "distractor_texts": distractor_texts,
        "created_at_utc": now_utc
    })
    gd_manifest[q.question_id] = now_utc.isoformat()

if new_pairs:
    df_pairs = spark.createDataFrame(new_pairs, schema=gd_pairs_schema)
    df_pairs.write.format("delta").mode("append").save(PATHS["gd_pairs"])
    save_manifest("processed_questions_gd", gd_manifest)
    logger.info(f"Appended {len(new_pairs)} pairs to Delta table.")

## Answer Generation Prompt

Defining the prompt template for generating factual answers based strictly on the provided context.

In [0]:
ANSWER_PROMPT_TEMPLATE = """
You are a factual answering system. Read the context below and answer the question based SOLELY on this context.
If the context does not contain enough information to answer the question, output exactly: "I cannot answer this question based on the provided context."

Output strictly valid JSON and nothing else. No markdown fences.
Format must be:
{{"answer": "..."}}

Context:
{golden_text}

Question:
{question_text}
"""

## LLM Client & Answer Generation Functions

Setting up the OpenAI-compatible client for NVIDIA NIM and implementing both single-pair and batch answer generation with automatic retry logic and fallback strategies.

In [0]:
openai_client = openai.OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=api
)

@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIError, openai.APITimeoutError)),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    stop=stop_after_attempt(5)
)
def call_nvidia_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 1024) -> str:
    response = openai_client.chat.completions.create(
        model=NVIDIA_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0.1
    )
    return response.choices[0].message.content

def generate_answer(question_text: str, golden_text: str) -> str:
    """Generate a single answer for one question-context pair."""
    prompt = ANSWER_PROMPT_TEMPLATE.format(golden_text=golden_text, question_text=question_text)
    system = "You are a strict JSON data generator."
    
    try:
        raw_output = call_nvidia_llm(prompt=prompt, system=system, max_tokens=1000)
    except Exception as e:
        logger.error(f"LLM API call failed: {e}")
        return None
        
    try:
        parsed = json.loads(raw_output)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
            except Exception as e:
                logger.error(f"Regex JSON extraction failed. Raw: {raw_output}. Error: {e}")
                return None
        else:
            logger.error(f"Failed to find JSON object in output. Raw: {raw_output}")
            return None
            
    if isinstance(parsed, dict) and "answer" in parsed:
        return parsed["answer"].strip()
    return None

BATCH_ANSWER_PROMPT_TEMPLATE = """
You are an expert question answering system. You will receive multiple question-context pairs, each labeled with a unique pair_id.
For EACH pair, generate a concise, direct answer based SOLELY on the provided context.
Do not use outside knowledge.

Output strictly valid JSON and nothing else. No markdown fences, no commentary.
Format must be a JSON object mapping pair_id to answer text:
{{
  "pair_id_1": "answer text here",
  "pair_id_2": "answer text here"
}}

Pairs:
{pairs_data}
"""

def generate_answers_for_batch(pair_batch: list, retry_smaller: bool = True) -> dict:
    """
    Generate answers for a batch of question-context pairs in a single API call.
    
    Args:
        pair_batch: List of dict objects with pair_id, question_text, golden_text
        retry_smaller: Whether to retry failed pairs in smaller batches
    
    Returns:
        Dict mapping pair_id -> answer text
    """
    if not pair_batch:
        return {}
    
    # Format all pairs into a single prompt
    pairs_data = ""
    for pair in pair_batch:
        pairs_data += f"""\n--- PAIR_ID: {pair['pair_id']} ---
Context: {pair['golden_text']}
Question: {pair['question_text']}
"""
    
    prompt = BATCH_ANSWER_PROMPT_TEMPLATE.format(pairs_data=pairs_data)
    system = "You are a strict JSON data generator."
    
    # Calculate token budget based on batch size
    prompt_tokens = len(prompt) // 4
    output_tokens = 150 * len(pair_batch)
    max_tokens = min(MAX_MODEL_TOKENS - prompt_tokens, output_tokens + 1000)
    
    try:
        raw_output = call_nvidia_llm(prompt=prompt, system=system, max_tokens=max_tokens)
    except Exception as e:
        logger.error(f"LLM API call failed for batch of {len(pair_batch)} pairs: {e}")
        return {}
    
    # Parse the LLM's JSON response with multiple fallback strategies
    parsed = None
    
    # Strategy 1: Direct JSON parsing
    try:
        parsed = json.loads(raw_output)
    except json.JSONDecodeError:
        pass
    
    # Strategy 2: Remove markdown code fences
    if parsed is None:
        cleaned = raw_output.strip()
        if cleaned.startswith('```'):
            lines = cleaned.split('\n')
            if lines[0].startswith('```'):
                lines = lines[1:]
            if lines and lines[-1].strip() == '```':
                lines = lines[:-1]
            cleaned = '\n'.join(lines)
            try:
                parsed = json.loads(cleaned)
            except json.JSONDecodeError:
                pass
    
    # Strategy 3: Extract JSON using regex
    if parsed is None:
        match = re.search(r'\{[\s\S]*\}', raw_output)
        if match:
            try:
                parsed = json.loads(match.group(0))
            except json.JSONDecodeError:
                logger.error(f"JSON parsing failed for batch of {len(pair_batch)} pairs")
        else:
            logger.error(f"No JSON object found in output")
    
    if parsed is None:
        sample = raw_output[:500] + '...' if len(raw_output) > 500 else raw_output
        logger.error(f"All JSON parsing attempts failed. Sample: {sample}")
        return {}
    
    # Validate the parsed structure
    if not isinstance(parsed, dict):
        logger.error(f"Expected dict, got {type(parsed)}")
        return {}
    
    # Extract and validate individual answers
    validated_results = {}
    for pair_id, answer_text in parsed.items():
        if isinstance(answer_text, str) and answer_text.strip():
            validated_results[pair_id] = answer_text.strip()
    
    # Retry any missing pairs in smaller batches
    successful_pairs = len(validated_results)
    total_pairs = len(pair_batch)
    
    if successful_pairs < total_pairs:
        missing_count = total_pairs - successful_pairs
        logger.warning(f"Batch incomplete: got {successful_pairs}/{total_pairs} answers ({missing_count} missing)")
        
        if retry_smaller and missing_count > 0 and missing_count < total_pairs:
            processed_pair_ids = set(validated_results.keys())
            missing_pairs = [p for p in pair_batch if p['pair_id'] not in processed_pair_ids]
            
            logger.info(f"Retrying {missing_count} missing pairs in smaller batches of {FALLBACK_BATCH_SIZE}...")
            
            for i in range(0, len(missing_pairs), FALLBACK_BATCH_SIZE):
                retry_batch = missing_pairs[i:i+FALLBACK_BATCH_SIZE]
                logger.info(f"  Retry sub-batch {i//FALLBACK_BATCH_SIZE + 1}: {len(retry_batch)} pairs")
                
                try:
                    retry_results = generate_answers_for_batch(retry_batch, retry_smaller=False)
                    validated_results.update(retry_results)
                    logger.info(f"    ✓ Recovered {len(retry_results)} pairs")
                except Exception as e:
                    logger.error(f"    ✗ Retry sub-batch failed: {e}")
    
    return validated_results

## Parallel Batch Answer Generation Pipeline

Processing all question-answer pairs using parallel batch API calls organized into waves to respect rate limits. This significantly speeds up answer generation while maintaining reliability.

In [0]:
import time

answers_schema = StructType([
    StructField("pair_id", StringType(), False),
    StructField("question_id", StringType(), False),
    StructField("answer_text", StringType(), False),
    StructField("model_used", StringType(), False),
    StructField("created_at_utc", TimestampType(), False)
])

# Initialize progress tracking
all_answers = []
failed_answers = 0
total_api_calls = 0

try:
    df_pairs_read = spark.read.format("delta").load(PATHS["gd_pairs"])
    pairs_local = df_pairs_read.select("pair_id", "question_id", "golden_text").collect()
except Exception as e:
    logger.info("Pairs table missing or empty.")
    pairs_local = []

ans_manifest = load_manifest("processed_pairs_ans")

# Identify pairs that still need answers
pending_pairs = []
q_dict = {q.question_id: q.question_text for q in questions_local}
for p in pairs_local:
    if p.pair_id not in ans_manifest:
        pending_pairs.append({
            "pair_id": p.pair_id,
            "question_id": p.question_id,
            "golden_text": p.golden_text,
            "question_text": q_dict.get(p.question_id, "")
        })

num_pending = len(pending_pairs)
logger.info(f"Found {num_pending} pairs needing answers.")

if num_pending == 0:
    logger.info("No pending pairs to process. All done!")
else:
    # Calculate processing strategy
    num_batches = (num_pending + BATCH_SIZE - 1) // BATCH_SIZE
    num_waves = (num_batches + API_RATE_LIMIT - 1) // API_RATE_LIMIT
    estimated_time_min = num_waves
    
    logger.info("="*70)
    logger.info(f"PARALLEL BATCH PROCESSING CONFIGURATION")
    logger.info(f"  Pairs per batch: {BATCH_SIZE}")
    logger.info(f"  Total batches: {num_batches}")
    logger.info(f"  Parallel calls per wave: {API_RATE_LIMIT}")
    logger.info(f"  Total waves: {num_waves}")
    logger.info(f"  Estimated time: ~{estimated_time_min} minutes")
    logger.info("="*70)
    
    start_time = time.time()
    
    def process_single_batch(batch_idx, pair_batch):
        """Process one batch of pairs in parallel."""
        batch_num = batch_idx + 1
        call_start = time.time()
        
        try:
            # Call LLM API for this batch
            batch_results = generate_answers_for_batch(pair_batch)
            call_duration = time.time() - call_start
            
            # Create answer records from results
            batch_answers = []
            now_utc = datetime.now(timezone.utc)
            failed_in_batch = 0
            
            for pair in pair_batch:
                pair_id = pair["pair_id"]
                question_id = pair["question_id"]
                
                if pair_id in batch_results:
                    batch_answers.append({
                        "pair_id": pair_id,
                        "question_id": question_id,
                        "answer_text": batch_results[pair_id],
                        "model_used": NVIDIA_MODEL,
                        "created_at_utc": now_utc
                    })
                else:
                    failed_in_batch += 1
            
            return {
                "success": True,
                "batch_num": batch_num,
                "answers": batch_answers,
                "pair_ids": [p["pair_id"] for p in pair_batch],
                "failed_pairs": failed_in_batch,
                "duration": call_duration,
                "timestamp": now_utc
            }
            
        except Exception as e:
            logger.error(f"Batch {batch_num} failed: {e}")
            return {
                "success": False,
                "batch_num": batch_num,
                "error": str(e),
                "pair_ids": [p["pair_id"] for p in pair_batch]
            }
    
    # Execute parallel processing in rate-limited waves
    for wave_idx in range(num_waves):
        wave_num = wave_idx + 1
        wave_start_time = time.time()
        
        # Calculate batch range for this wave
        wave_start_batch = wave_idx * API_RATE_LIMIT
        wave_end_batch = min(wave_start_batch + API_RATE_LIMIT, num_batches)
        batches_in_wave = wave_end_batch - wave_start_batch
        
        logger.info(f"\n{'='*70}")
        logger.info(f"WAVE {wave_num}/{num_waves} - Processing {batches_in_wave} batches in parallel")
        logger.info(f"  (Each batch = {BATCH_SIZE} pairs, Total pairs this wave = {batches_in_wave * BATCH_SIZE})")
        logger.info(f"{'='*70}")
        
        # Submit all batches in parallel
        wave_futures = []
        with ThreadPoolExecutor(max_workers=API_RATE_LIMIT) as executor:
            for batch_idx in range(wave_start_batch, wave_end_batch):
                batch_start_idx = batch_idx * BATCH_SIZE
                batch_end_idx = min(batch_start_idx + BATCH_SIZE, num_pending)
                pair_batch = pending_pairs[batch_start_idx:batch_end_idx]
                
                # Submit this batch to the thread pool
                future = executor.submit(process_single_batch, batch_idx, pair_batch)
                wave_futures.append(future)
            
            logger.info(f"✓ Submitted {len(wave_futures)} parallel API calls at t=0s")
            logger.info("Processing responses as they arrive during 60s window...")
            
            # Process results AS THEY ARRIVE during the 60s window
            wave_answers = []
            completed = 0
            failed_batches = 0
            
            for future in as_completed(wave_futures):
                completed += 1
                result = future.result()
                total_api_calls += 1
                elapsed = time.time() - wave_start_time
                
                if result["success"]:
                    # Add answers to wave collection
                    wave_answers.extend(result["answers"])
                    
                    # Update manifest for processed pairs
                    for pair_id in result["pair_ids"]:
                        if pair_id in [a["pair_id"] for a in result["answers"]]:
                            ans_manifest[pair_id] = result["timestamp"].isoformat()
                    
                    failed_answers += result["failed_pairs"]
                    
                    logger.info(
                        f"  ✓ Batch {result['batch_num']}: "
                        f"{len(result['answers'])} answers "
                        f"({result['duration']:.2f}s) "
                        f"[{completed}/{batches_in_wave} complete @ t={elapsed:.1f}s]"
                    )
                else:
                    failed_batches += 1
                    failed_answers += len(result["pair_ids"])
                    logger.error(f"  ✗ Batch {result['batch_num']} failed: {result['error']}")
            
            # All responses collected - calculate actual wave duration
            wave_duration = time.time() - wave_start_time
            logger.info(f"\n✓ Wave {wave_num} responses collected in {wave_duration:.2f}s")
            logger.info(f"  Answers generated: {len(wave_answers)}")
            logger.info(f"  Failed batches: {failed_batches}")
            
            # Append wave answers to master list
            all_answers.extend(wave_answers)
            
            # Save wave answers to Delta table (during remaining wait time)
            if wave_answers:
                df_wave_ans = spark.createDataFrame(wave_answers, schema=answers_schema)
                df_wave_ans.write.format("delta").mode("append").save(PATHS["answers"])
                save_manifest("processed_pairs_ans", ans_manifest)
                logger.info(f"  ✓ Saved {len(wave_answers)} answers to Delta table")
            
            # Progress update
            elapsed_min = (time.time() - start_time) / 60.0
            progress_pct = (wave_num / num_waves) * 100
            total_answers_so_far = len(all_answers)
            logger.info(f"\nOverall Progress: {progress_pct:.1f}% | Elapsed: {elapsed_min:.2f}min | Total Answers: {total_answers_so_far}")
        
        # Rate limiting: wait for REMAINING time in 60s window
        if wave_num < num_waves:
            elapsed = time.time() - wave_start_time
            remaining = MIN_WAVE_INTERVAL - elapsed
            if remaining > 0:
                logger.info(f"Waiting {remaining:.1f}s to complete 60s window before next wave...")
                time.sleep(remaining)
            else:
                logger.info(f"Wave took {elapsed:.1f}s (exceeded 60s window), starting next wave immediately")
    
    # ===== RETRY PASS: Catch stragglers =====
    failed_pair_ids = [p["pair_id"] for p in pending_pairs if p["pair_id"] not in ans_manifest]
    
    if failed_pair_ids:
        retry_pairs = [p for p in pending_pairs if p["pair_id"] in failed_pair_ids]
        num_retry = len(retry_pairs)
        
        logger.info(f"\n{'='*70}")
        logger.info(f"RETRY PASS - Attempting to recover {num_retry} failed pairs")
        logger.info(f"{'='*70}")
        
        # Calculate retry batching
        RETRY_BATCH_SIZE = FALLBACK_BATCH_SIZE
        RETRY_WORKERS = min(10, API_RATE_LIMIT)
        num_retry_batches = (num_retry + RETRY_BATCH_SIZE - 1) // RETRY_BATCH_SIZE
        num_retry_waves = (num_retry_batches + RETRY_WORKERS - 1) // RETRY_WORKERS
        
        logger.info(f"  Retry batch size: {RETRY_BATCH_SIZE}")
        logger.info(f"  Retry workers: {RETRY_WORKERS}")
        logger.info(f"  Retry waves: {num_retry_waves}")
        
        retry_recovered = []
        retry_failed = 0
        
        for retry_wave_idx in range(num_retry_waves):
            retry_wave_num = retry_wave_idx + 1
            retry_wave_start_time = time.time()
            
            wave_start_batch = retry_wave_idx * RETRY_WORKERS
            wave_end_batch = min(wave_start_batch + RETRY_WORKERS, num_retry_batches)
            batches_in_retry_wave = wave_end_batch - wave_start_batch
            
            logger.info(f"\n  Retry Wave {retry_wave_num}/{num_retry_waves} - Processing {batches_in_retry_wave} batches")
            
            retry_wave_answers = []
            retry_wave_futures = []
            
            with ThreadPoolExecutor(max_workers=RETRY_WORKERS) as executor:
                for batch_idx in range(wave_start_batch, wave_end_batch):
                    batch_start_idx = batch_idx * RETRY_BATCH_SIZE
                    batch_end_idx = min(batch_start_idx + RETRY_BATCH_SIZE, num_retry)
                    retry_batch = retry_pairs[batch_start_idx:batch_end_idx]
                    
                    future = executor.submit(process_single_batch, batch_idx, retry_batch)
                    retry_wave_futures.append(future)
                
                completed = 0
                for future in as_completed(retry_wave_futures):
                    completed += 1
                    result = future.result()
                    
                    if result["success"]:
                        retry_wave_answers.extend(result["answers"])
                        for pair_id in result["pair_ids"]:
                            if pair_id in [a["pair_id"] for a in result["answers"]]:
                                ans_manifest[pair_id] = result["timestamp"].isoformat()
                        logger.info(f"    ✓ RECOVERED Batch {result['batch_num']}: {len(result['answers'])} answers ({result['duration']:.2f}s)")
                    else:
                        retry_failed += len(result["pair_ids"])
                        logger.error(f"    ✗ FAILED AGAIN Batch {result['batch_num']}: {result['error']}")
                
                retry_wave_duration = time.time() - retry_wave_start_time
                logger.info(f"  Retry Wave {retry_wave_num} completed in {retry_wave_duration:.2f}s")
                logger.info(f"    Recovered: {len(retry_wave_answers)} | Still Failed: {retry_failed}")
                
                # Save recovered answers
                retry_recovered.extend(retry_wave_answers)
                if retry_wave_answers:
                    df_retry_ans = spark.createDataFrame(retry_wave_answers, schema=answers_schema)
                    df_retry_ans.write.format("delta").mode("append").save(PATHS["answers"])
                    save_manifest("processed_pairs_ans", ans_manifest)
                    logger.info(f"    ✓ Saved {len(retry_wave_answers)} recovered answers to Delta table")
            
            # Delay between retry waves
            if retry_wave_num < num_retry_waves:
                logger.info(f"  Waiting {MIN_WAVE_INTERVAL}s before next retry wave...")
                time.sleep(MIN_WAVE_INTERVAL)
        
        # Update totals with retry results
        all_answers.extend(retry_recovered)
        failed_answers = retry_failed
        
        logger.info(f"\n{'='*70}")
        logger.info(f"RETRY PASS COMPLETE")
        logger.info(f"  Recovered: {len(retry_recovered)}")
        logger.info(f"  Permanently Failed: {failed_answers}")
        logger.info(f"={'='*70}")
    
    # Final summary
    total_time = time.time() - start_time
    success_rate = (len(all_answers)/(len(all_answers)+failed_answers)*100) if (len(all_answers)+failed_answers) > 0 else 0
    
    logger.info(f"\n{'='*70}")
    logger.info(f"ANSWER GENERATION COMPLETE")
    logger.info(f"  Total pairs processed: {num_pending}")
    logger.info(f"  ✓ Successful answers: {len(all_answers)}")
    logger.info(f"  ✗ Permanently failed: {failed_answers}")
    logger.info(f"  Success rate: {success_rate:.1f}%")
    logger.info(f"  Total time: {total_time/60:.2f} minutes")
    logger.info(f"  Avg time per answer: {total_time/max(1,len(all_answers)):.2f}s")
    logger.info(f"  Total API calls: {total_api_calls}")
    
    if failed_answers == 0:
        logger.info(f"  🎉 100% COVERAGE - No missed answers!")
    else:
        logger.warning(f"  ⚠️  {failed_answers} pairs still failed after retry pass")
    
    logger.info(f"={'='*70}")

new_answers = all_answers

## Join Data & Create Final Training Dataset

Combining questions, contexts, and answers into complete training triplets ready for export.

In [0]:
try:
    df_q = spark.read.format("delta").load(PATHS["questions"])
    df_p = spark.read.format("delta").load(PATHS["gd_pairs"])
    df_a = spark.read.format("delta").load(PATHS["answers"])
    
    # Join questions, pairs, and answers
    df_joined = df_q.alias("q") \
        .join(df_p.alias("p"), "question_id") \
        .join(df_a.alias("a"), ["pair_id", "question_id"]) \
        .select(
            F.col("pair_id").alias("id"),
            F.col("q.question_text").alias("question"),
            F.col("a.answer_text").alias("answer"),
            F.col("p.golden_text").alias("positive_ctx"),
            F.col("p.distractor_texts").alias("negative_ctxs"),
            F.struct(
                F.col("q.doc_id"),
                F.col("q.chunk_id"),
                F.col("a.model_used"),
                F.col("a.created_at_utc")
            ).alias("metadata")
        )
except Exception as e:
    logger.error(f"Join step failed: {e}")
    df_joined = None

## Export to JSONL Format

Writing the final dataset to a JSONL file, with each line containing a complete training example.

In [0]:
jsonl_path = ""
line_count = 0

if df_joined:
    run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    filename = f"dataset_{run_ts}.jsonl"
    ensure_dir(PATHS["datasets"])
    jsonl_path = os.path.join(PATHS["datasets"], filename)
    
    logger.info(f"Writing final dataset to {jsonl_path}...")
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for row in df_joined.toLocalIterator():
            record = {
                "id": row.id,
                "question": row.question,
                "answer": row.answer,
                "positive_ctx": row.positive_ctx,
                "negative_ctxs": row.negative_ctxs,
                "metadata": {
                    "doc_id": row.metadata.doc_id,
                    "chunk_id": row.metadata.chunk_id,
                    "model_used": row.metadata.model_used,
                    "created_at_utc": row.metadata.created_at_utc.isoformat() if row.metadata.created_at_utc else None
                }
            }
            f.write(json.dumps(record) + "\n")
            line_count += 1
            
    logger.info(f"Successfully wrote {line_count} records to JSONL.")

## Validate Dataset Quality

Verifying the exported JSONL file has correct structure and all required fields.

In [0]:
if line_count > 0 and os.path.exists(jsonl_path):
    with open(jsonl_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
    assert len(lines) == line_count, f"Line count mismatch. Expected {line_count}, got {len(lines)}"
    
    sample_lines = lines[:3] + lines[-3:] if len(lines) >= 6 else lines
    for i, line in enumerate(sample_lines):
        record = json.loads(line)
        for key in ["question", "answer", "positive_ctx", "negative_ctxs"]:
            assert key in record, f"Missing required key '{key}' in record {i}"
    
    logger.info("JSONL schema validation passed successfully.")

## Pipeline Execution Summary

Displaying final statistics and output file location.

In [0]:
print("="*40)
print("RUN SUMMARY: NOTEBOOK 2")
print(f"Golden/Distractor Pairs Built: {len(new_pairs)}")
print(f"Answers Generated Successfully: {len(new_answers)}")
print(f"Answer Failures: {failed_answers}")
if df_joined:
    print(f"Final Dataset Exported To: {jsonl_path}")
    print(f"Total Triplets Exported: {line_count}")
print("="*40)